# 03 | Tool-calling demand intelligence agent

Run after notebooks 01 and 02. The saved forecast covers October 4–10, 2026, and is a historical demonstration, not a live forecast. A compatible Databricks model-serving endpoint and notebook-context authentication are required.

**Security:** Do not print or commit tokens, private sales data, or notebook outputs.

In [ ]:
%pip install openai

In [ ]:
from openai import OpenAI

# Databricks notebook context token. Never print, export, or commit it.
token = (
    dbutils.notebook.entry_point.getDbutils()
    .notebook().getContext().apiToken().get()
)
host = spark.conf.get("spark.databricks.workspaceUrl")
client = OpenAI(api_key=token, base_url=f"https://{host}/ai-gateway/mlflow/v1")
MODEL = "system.ai.gpt-oss-120b"
print("LLM client configured")

In [ ]:
# Databricks tools for the AI agent

FORECAST_TABLE = "workspace.default.restaurant_forecast"
HISTORY_TABLE = "workspace.default.restaurant_daily_demand"



def get_demand_forecast():
    """Retrieve forecasts with verified calendar dates."""

    rows = spark.sql(f"""
        SELECT
            business_day,
            DATE_FORMAT(business_day, 'EEEE') AS weekday,
            predicted_orders
        FROM {FORECAST_TABLE}
        ORDER BY business_day
    """).collect()

    return [
        {
            "date": str(row["business_day"]),
            "weekday": row["weekday"],
            "predicted_orders": int(row["predicted_orders"])
        }
        for row in rows
    ]

def get_historical_demand():
    """Retrieve average historical demand by weekday."""

    rows = spark.sql(f"""
        SELECT
            DATE_FORMAT(business_day, 'EEEE') AS weekday,
            DAYOFWEEK(business_day) AS weekday_number,
            ROUND(AVG(orders), 2) AS avg_orders,
            COUNT(*) AS recorded_days
        FROM {HISTORY_TABLE}
        GROUP BY
            DATE_FORMAT(business_day, 'EEEE'),
            DAYOFWEEK(business_day)
        ORDER BY weekday_number
    """).collect()

    return [
        {
            "weekday": row["weekday"],
            "average_orders": float(row["avg_orders"]),
            "recorded_days": int(row["recorded_days"])
        }
        for row in rows
    ]


print("Both functions defined successfully!")

In [ ]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_demand_forecast",
            "description": (
                "Retrieve the saved restaurant order forecast "
                "for October 4 through October 10, 2026. "
                "These are historical demonstration forecasts, "
                "not live forecasts."
            ),
            "parameters": {
                "type": "object",
                "properties": {},
                "required": []
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_historical_demand",
            "description": (
                "Retrieve historical average restaurant order "
                "counts by weekday from Databricks SQL."
            ),
            "parameters": {
                "type": "object",
                "properties": {},
                "required": []
            }
        }
    }
]

tool_functions = {
    "get_demand_forecast": get_demand_forecast,
    "get_historical_demand": get_historical_demand
}

print("Two tools registered successfully.")

In [ ]:
import json


SYSTEM_PROMPT = """
You are a restaurant demand intelligence agent.

You answer operational questions using Databricks tools.

MANDATORY RULES:

1. Always retrieve data using tools before answering
   numerical questions.

2. For questions comparing forecasts with historical
   demand, you MUST call BOTH:
   - get_demand_forecast
   - get_historical_demand

3. Never invent, modify, or reinterpret dates.

4. Copy dates and weekdays exactly as returned by tools.

5. Do not claim a weekday unless it is provided by
   the tool result.

6. Use only the numerical values returned by tools.

7. If a user asks for a comparison, calculate the
   difference using the retrieved values.

8. Monday is normally a restaurant closure day.

9. Forecasts cover October 4-10, 2026 only.
   They are historical demonstration forecasts.

10. Do not invent staffing levels, savings, or
    profitability improvements.

11. If the requested forecast is unavailable,
    explain the limitation clearly.

12. Answer the user's actual question directly.
    Do not substitute a different analysis.
"""


SYSTEM_PROMPT += """

OPERATIONAL RECOMMENDATION GUARDRAILS:

- You have demand data only.
- You do not have staffing schedules, kitchen capacity,
  inventory levels, or supplier information.
- Never claim staffing is sufficient or insufficient.
- Never claim there is no capacity risk.
- Never prescribe exact inventory quantities.
- Clearly distinguish observations from recommendations.
- When relevant data is unavailable, explicitly say so.
- Recommendations should be conditional and actionable.
- Do not imply the forecast is guaranteed to be accurate.
"""


def ask_agent(question):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question}
    ]

    for step in range(5):
        response = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=tools,
            tool_choice="auto",
            max_tokens=1000
        )

        message = response.choices[0].message
        messages.append(message.model_dump(exclude_none=True))

        if not message.tool_calls:
            return message.content

        for tool_call in message.tool_calls:
            name = tool_call.function.name

            if name not in tool_functions:
                result = {"error": "Unknown tool"}
            else:
                try:
                    args = json.loads(
                        tool_call.function.arguments or "{}"
                    )

                    if args:
                        result = {
                            "error": "This tool takes no arguments"
                        }
                    else:
                        result = tool_functions[name]()

                    print(f"Tool executed: {name}")

                except Exception as e:
                    result = {
                        "error": type(e).__name__,
                        "message": str(e)[:200]
                    }

            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": json.dumps(result)
            })

    return "Agent stopped after reaching its tool-call limit."

In [ ]:
question = """
How many orders are predicted for Friday,
October 9, 2026?

Compare this with the historical average for Fridays.
Explain the operational implications.
"""

print(ask_agent(question))

In [ ]:
print(ask_agent(
    "How many orders will Shiraz receive on "
    "November 15, 2026?"
))

In [ ]:
print(ask_agent(
    "What is the total predicted demand for "
    "October 4 through October 10, 2026? "
    "Which day has the highest forecast?"
))